# Spider/Radar Performance Plot

Create radar plot comparing trimodal and three bimodal_matching models across all metrics grouped by modality pairing

In [ ]:
# Parameters from Snakemake
model_configs = snakemake.params.model_configs
metrics_by_modality = snakemake.params.metrics_by_modality

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
from pathlib import Path
from math import pi

# Apply matplotlib style
plt.style.use(snakemake.input.mpl_style)

In [ ]:
# Load data for all 4 models
model_data = {}

for i, (model_name, dataset_combo) in enumerate(model_configs):
    data = {}

    # HEST results - load per-tissue/dataset performance
    with open(snakemake.input.hest_results[i]) as f:
        hest_data = json.load(f)
        # Load individual dataset performances instead of overall_performance
        for dataset_entry in hest_data.get("datasets", []):
            dataset_name = dataset_entry["dataset_name"]
            metric_key = "test_retrieval/transcriptome_image/rocauc_macroAvg"
            if metric_key in dataset_entry:
                data[f"hest_{dataset_name}"] = dataset_entry[metric_key]

    # MUSK results
    with open(snakemake.input.musk_results[i]) as f:
        musk_data = json.load(f)
        data["pannuke"] = musk_data["task_summaries"]["zeroshot_classification"][
            "balanced_acc"
        ]["pannuke"]
        data["skin"] = musk_data["task_summaries"]["zeroshot_classification"][
            "balanced_acc"
        ]["skin"]

    # CellWhisperer results
    cw_df = pd.read_csv(snakemake.input.cwevals_results[i], index_col=0)
    for metric in cw_df.index:
        data[metric] = cw_df.iloc[cw_df.index == metric, 0].values[0]

    # Lung results
    lung_df = pd.read_csv(snakemake.input.lung_results[i])
    for _, row in lung_df.iterrows():
        prefix = f"{row['dataset']}_{row['metadata_col']}"
        data[f"{prefix}_accuracy"] = row["accuracy"]

    # PathoCellBench results
    with open(snakemake.input.pathocell_results[i]) as f:
        pathocell_data = json.load(f)
        spider_metrics = pathocell_data.get("spider_plot_metrics", {})
        print(f"Loading PathoCellBench data for {model_name}: {spider_metrics}")
        for metric, value in spider_metrics.items():
            data[metric] = value

    model_data[model_name] = data

In [ ]:
# Prepare data for radar plot
all_metrics = []
modality_labels = []
modality_colors = snakemake.params.modality_colors


for modality, metrics in metrics_by_modality.items():
    all_metrics.extend(metrics)
    modality_labels.extend([modality] * len(metrics))

# Create DataFrame with all model performances
radar_data = pd.DataFrame(index=all_metrics)
for model_name, data in model_data.items():
    radar_data[model_name] = [data.get(metric, 0) for metric in all_metrics]

# Normalize data to 0-1 scale for better visualization
radar_data_norm = radar_data.copy()
for metric in radar_data.index:
    max_val = radar_data.loc[metric].max()
    min_val = radar_data.loc[metric].min()
    if max_val > min_val:
        radar_data_norm.loc[metric] = (radar_data.loc[metric] - min_val) / (
            max_val - min_val
        )
    else:
        radar_data_norm.loc[metric] = 0.5

In [ ]:
# Create radar plot
N = len(all_metrics)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the circle

fig, ax = plt.subplots(figsize=(6, 5), subplot_kw=dict(projection="polar"))

# Colors for each model (trimodal, text-transcriptome, image-transcriptome, text-image)
model_colors = [
    modality_colors["trimodal"],
    modality_colors["text-transcriptome"],
    modality_colors["image-transcriptome"],
    modality_colors["text-image"],
]

# Plot each model
for i, (model_name, color) in enumerate(zip(radar_data_norm.columns, model_colors)):
    values = radar_data_norm[model_name].tolist()
    values += values[:1]  # Complete the circle

    ax.plot(angles, values, "o-", linewidth=2, label=model_name, color=color)
    ax.fill(angles, values, alpha=0.1, color=color)

# Add metric labels with modality color coding
ax.set_xticks(angles[:-1])
metric_labels = [
    metric.replace("valfn_zshot_TabSap_", "").replace("/", "\n")
    for metric in all_metrics
]
ax.set_xticklabels(metric_labels, fontsize=10)

# Color code the metric labels by modality
for i, (angle, label, modality) in enumerate(
    zip(angles[:-1], ax.get_xticklabels(), modality_labels)
):
    label.set_color(modality_colors[modality])
    label.set_fontweight("bold")

# Customize plot
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=8)
ax.grid(True, alpha=0.3)

# Add model legend
from matplotlib.patches import Patch

model_legend = [
    Patch(facecolor=color, label=model)
    for model, color in zip(radar_data_norm.columns, model_colors)
]
ax.legend(
    handles=model_legend,
    loc="upper right",
    bbox_to_anchor=(1.3, 1.0),
    fontsize=12,
    title="Models",
)


plt.title(
    "Model Performance Comparison Across Modality Pairings",
    size=16,
    fontweight="bold",
    pad=20,
)

# Save plot
plt.tight_layout()
plt.savefig(snakemake.output.plot, dpi=300, bbox_inches="tight")
plt.savefig(snakemake.output.plot + ".svg", dpi=300, bbox_inches="tight")
plt.show()